# Selective RAD kernels and conditioning

The five coefficient cases follow Araya et al. (2024), section 5.3. The recorded spectra use physical nodal and constant bases before solver equilibration. Published marker uncertainty, geometric choices and any uncertified solves remain explicit.

In [ ]:
from pathlib import Path
import sys

# Locate the checkout from the repository root or any notebook subdirectory.
ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").is_file()
     and (path / "src" / "pymhm").is_dir()
     and (path / "notebooks").is_dir()),
    None,
)
if ROOT is None:
    raise RuntimeError("Run this notebook from the PyMHM checkout.")
root = ROOT
for directory in (ROOT, ROOT / "examples"):
    if str(directory) not in sys.path:
        sys.path.insert(0, str(directory))


In [ ]:
from pathlib import Path
import json
from IPython.display import Image, display
record = json.loads((root / "examples/results/rad-conditioning.json").read_text())
rows = record["records"]
assert set(r["case"] for r in rows) == {1, 2, 3, 4, 5}
[(s, sum(r["study"] == s for r in rows)) for s in ("epsilon", "omega", "local", "skeleton", "quadrature")]

In [ ]:
[(r["case"], r["kernel_cells"], r["global_condition"]) for r in rows
 if r["study"] == "epsilon" and r["epsilon"] == 1.0]

The local ordinates differ from the published values. A matrix condition number does not certify a reconstructed solution: records that fail the unchanged residual criterion retain `not_certified` status.

In [ ]:
[(r["study"], r["case"], r["epsilon"], r["omega"]) for r in rows if r["status"] != "verified"]

In [ ]:
for name in ("epsilon", "omega", "mesh"):
    display(Image(filename=str(root / f"docs/figures/rad-conditioning/{name}.png")))

Independent DOLFINx/UFL complete-system comparisons use the same material, boundary conditions and local/skeletal spaces. Saved coordinates, connectivity and polynomial ordering permit field replay. The native code does not reuse PyMHM local matrices or condensation. Historical connectivity remains unspecified.

In [ ]:
import hashlib
native = json.loads((root / "examples/results/rad-native/verification.json").read_text())
assert native["accepted"]
native_rows = [row for row in native["rows"] if row["kind"] == "conditioning"]
assert len(native_rows) == 10
for row in native_rows:
    archive = root / "examples/results/rad-native" / row["archive"]
    assert hashlib.sha256(archive.read_bytes()).hexdigest() == row["archive_sha256"]
    assert row["accepted"]
[(row["n"], row["local_refinement"], row["case"], row["epsilon"],
  row["physical_norms"]["20"]["physical_flux"]["relative_difference"])
 for row in native_rows]